In [2]:
# Install required libraries for lexical retrieval and Ollama
!pip install rank_bm25 scikit-learn ollama


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Import necessary libraries
import re
import json
import os
import ollama
from rank_bm25 import BM25Okapi
from typing import List, Tuple

In [4]:
# Load chunked snippets generated by Module 1
def load_snippets(load_path="./output/snippets.json"):
    try:
        with open(load_path, "r", encoding="utf-8") as f:
            snippets = json.load(f)
        print(f"Successfully loaded {len(snippets)} snippets")
        return snippets
    except Exception as e:
        print(f"Failed to load snippets: {e}")
        return []

# Execute loading
snippets = load_snippets()

Successfully loaded 32 snippets


In [5]:
# Clean and tokenize input text
def preprocess_text(text: str) -> List[str]:
    text = re.sub(r'[^\w\s]', ' ', text.lower())
    return text.split()

In [6]:
# BM25-based lexical search
def lexical_search(query: str, snippets: List[dict], top_k: int = 3) -> List[Tuple[float, List[str], int]]:
    snippet_texts = [s["text"] for s in snippets]
    tokenized_texts = [preprocess_text(t) for t in snippet_texts]
    tokenized_query = preprocess_text(query)

    bm25 = BM25Okapi(tokenized_texts)
    scores = bm25.get_scores(tokenized_query)

    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

    results = []
    for idx in top_indices:
        overlap_keywords = list(set(tokenized_query) & set(tokenized_texts[idx]))
        results.append((round(scores[idx], 4), overlap_keywords, idx))
    return results

In [7]:
# Ollama raw generation (matches your template: raw=True)
MODEL = "gemma3:4b"

def ollama_generate(prompt: str) -> str:
    response = ollama.generate(
        model=MODEL,
        prompt=prompt,
        stream=False,
        raw=True,
        options={
            "num_predict": 180,
            "temperature": 0.3
        }
    )
    return response["response"].strip()

In [8]:
# Full lexical RAG pipeline: retrieve → prompt → generate
def answer_with_lexical_rag(query: str, snippets: List[dict], top_k: int = 3) -> str:
    search_results = lexical_search(query, snippets, top_k)
    if not search_results:
        return "No relevant information found."

    context_parts = []
    for i, (score, kw, idx) in enumerate(search_results):
        context_parts.append(f"[Snippet {i+1}] {snippets[idx]['text']}")
    context = "\n\n".join(context_parts)

    prompt = f"""Use only the context below to answer the question.
Cite snippet numbers like [1][2].

Context:
{context}

Question: {query}
Answer:"""

    return ollama_generate(prompt)

In [9]:
# Test the full pipeline
if __name__ == "__main__":
    test_query = "What is academic advising?"

    print("\n=== Lexical Search Result ===")
    search_result = lexical_search(test_query, snippets)
    for score, overlap, idx in search_result:
        print(f"Score: {score} | Keywords: {overlap} | Index: {idx}")

    print("\n=== Lexical RAG Answer ===")
    answer = answer_with_lexical_rag(test_query, snippets)
    print(answer)


=== Lexical Search Result ===
Score: 5.7348 | Keywords: ['what', 'is', 'academic'] | Index: 15
Score: 5.5427 | Keywords: ['what', 'is'] | Index: 14
Score: 4.6727 | Keywords: ['advising', 'academic'] | Index: 0

=== Lexical RAG Answer ===
According to the context, academic advising helps new students smoothly adapt to university study life. It provides guidance on major selection, effective study skills development and phased study plan formulation. It also offers ongoing support for students to update study plans, connect academic learning with post-graduation development, and set clear career goals or further study plans. [1][2][3]


In [11]:
# ### Module 2 Validation
# %%
# Final validation
if snippets:
    print("Module 2 validation passed!")
    print(f"Total snippets available: {len(snippets)}")
    print("lexical_search() and answer_with_lexical_rag() are ready.")
else:
    print("Validation failed: please check snippets.json from Module1.")

Module 2 validation passed!
Total snippets available: 32
lexical_search() and answer_with_lexical_rag() are ready.
